# Hands-on CDSE — sentinelhub-py

The same evalscripts you ran in the Copernicus Browser, now from Python.

## §1 — Authenticate

CDSE runs its own Sentinel Hub deployment, so point `SHConfig` at the CDSE endpoints.

In [ ]:
from sentinelhub import SHConfig

config = SHConfig()
# config.sh_client_id = "..."
# config.sh_client_secret = "..."
# or put them in ~/.config/sentinelhub/config.toml

config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = (
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE"
    "/protocol/openid-connect/token"
)
print("configured:", config.sh_base_url)

## §2 — Area, time, collection

In [ ]:
from sentinelhub import BBox, CRS, DataCollection, bbox_to_dimensions

# Hiroshima delta, ~9 x 10 km. (west, south, east, north) in WGS84.
aoi = BBox((132.40, 34.33, 132.50, 34.42), crs=CRS.WGS84)
resolution = 10  # m
size = bbox_to_dimensions(aoi, resolution=resolution)

s2l2a = DataCollection.SENTINEL2_L2A.define_from("s2l2a", service_url=config.sh_base_url)

print("size (px):", size)  # 2500 max per side

## §3 — The evalscript

The same NDVI script from Exercise 2, returned as FLOAT32.

In [4]:
evalscript_ndvi = """
//VERSION=3
function setup() {
  return { input: ["B04", "B08"], output: { bands: 1, sampleType: "FLOAT32" } };
}
function evaluatePixel(s) {
  return [(s.B08 - s.B04) / (s.B08 + s.B04)];
}
"""

## §4 — Process API

The script runs server-side; what comes back is the array, not an image to download.

In [ ]:
import matplotlib.pyplot as plt
from sentinelhub import SentinelHubRequest, MimeType

request = SentinelHubRequest(
    evalscript=evalscript_ndvi,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=s2l2a,
            time_interval=("2026-06-01", "2026-07-31"),
            mosaicking_order="leastCC",  # API-only; the Browser offers most/least recent
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=aoi,
    size=size,
    config=config,
)

ndvi = request.get_data()[0]
print("shape:", ndvi.shape, "range:", float(ndvi.min()), "to", float(ndvi.max()))

plt.figure(figsize=(6, 6))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
plt.colorbar(label="NDVI"); plt.title("NDVI — least-cloudy, Jun–Jul 2026"); plt.axis("off")

## §5 — Statistical API

Aggregated numbers instead of pixels. `dataMask` decides what gets counted.

In [ ]:
from sentinelhub import SentinelHubStatistical

evalscript_stat = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ["B04", "B08", "SCL", "dataMask"] }],
    output: [
      { id: "ndvi", bands: 1, sampleType: "FLOAT32" },
      { id: "dataMask", bands: 1 }
    ]
  };
}
function evaluatePixel(s) {
  let ndvi = (s.B08 - s.B04) / (s.B08 + s.B04);
  // Which SCL classes to drop is a judgement call, not a fact. The Browser's
  // NDVI preset drops [8,9,10] and keeps cloud shadow; this drops shadow too.
  let valid = (s.dataMask === 1 && ![3, 8, 9, 10, 11].includes(s.SCL)) ? 1 : 0;
  return { ndvi: [ndvi], dataMask: [valid] };
}
"""

stat_request = SentinelHubStatistical(
    aggregation=SentinelHubStatistical.aggregation(
        evalscript=evalscript_stat,
        time_interval=("2025-08-01", "2026-07-31"),
        aggregation_interval="P1M",
        # Pixels, not resolution: resx/resy are in bbox-CRS units, so on a WGS84
        # box resolution=(10, 10) would ask for ten degrees per pixel.
        # 60 m is plenty for a mean over the whole AOI and costs ~2 PU instead
        # of ~84 for the year.
        size=bbox_to_dimensions(aoi, resolution=60),
    ),
    input_data=[SentinelHubStatistical.input_data(s2l2a)],
    bbox=aoi,
    config=config,
)

stats = stat_request.get_data()[0]

In [ ]:
# A failed interval carries `error` and no `outputs`, and empty ones are dropped
# entirely, so don't index blind.
dates, means, skipped = [], [], []
for item in stats["data"]:
    if "outputs" not in item:
        skipped.append(item.get("interval", {}).get("from", "?")[:10])
        continue
    band = item["outputs"]["ndvi"]["bands"]["B0"]["stats"]
    if band["sampleCount"] <= band["noDataCount"]:
        skipped.append(item["interval"]["from"][:10])
        continue
    dates.append(item["interval"]["from"][:10])
    means.append(band["mean"])

print(f"{len(dates)} intervals, {len(skipped)} skipped: {skipped}")

plt.figure(figsize=(9, 4))
plt.plot(dates, means, marker="o")
plt.title("Mean NDVI, trailing 12 months"); plt.ylabel("NDVI")
plt.xticks(rotation=45, ha="right"); plt.grid(alpha=0.3); plt.tight_layout()

## §6 — Ten lakes, one year

The Statistical API takes one geometry per request, so ten lakes is ten requests.
Collection is CLMS Lake Surface Water Temperature, 1 km, 10-daily.

In [ ]:
from sentinelhub import Geometry

lswt = DataCollection.define_byoc(
    collection_id="401ca642-a169-4783-b1cf-cbd33e98eccb", name="lswt"
)

# LSWT is INT16 Kelvin, scaling 0.01, offset 273.15 — so Celsius is just raw * 0.01.
# CLMS has no SCL; quality arrives as QLEVEL, where >= 4 is acceptable or better.
evalscript_lswt = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ["LSWT", "QLEVEL", "dataMask"] }],
    output: [
      { id: "celsius",  bands: 1, sampleType: "FLOAT32" },
      { id: "dataMask", bands: 1 }
    ]
  };
}
function evaluatePixel(s) {
  var valid = (s.dataMask === 1 && s.QLEVEL >= 4) ? 1 : 0;
  return { celsius: [s.LSWT * 0.01], dataMask: [valid] };
}
"""

In [ ]:
# Ten lakes chosen for curve diversity, not coverage: latitude 69N to 39S, both
# hemispheres, and Balaton vs Geneva at the same latitude (shallow swings, deep
# damps). Boxes sit on open water — at 1 km a shoreline pixel drags the mean.
LAKES = [
    ("Inari (FI)",        68.95,  27.70), ("Ladoga (RU)",       61.00,  31.50),
    ("Balaton (HU)",      46.85,  17.75), ("Geneva (CH)",       46.45,   6.55),
    ("Biwa (JP)",         35.30, 136.15), ("Taihu (CN)",        31.20, 120.20),
    ("Nasser (EG)",       23.00,  32.80), ("Victoria (UG)",     -1.00,  33.00),
    ("Titicaca (PE/BO)", -15.85, -69.35), ("Taupo (NZ)",       -38.80, 175.90),
]

def box(lat, lon, half_deg=0.03):
    """Small square polygon around a point, as GeoJSON."""
    return {"type": "Polygon", "coordinates": [[
        [lon - half_deg, lat - half_deg], [lon + half_deg, lat - half_deg],
        [lon + half_deg, lat + half_deg], [lon - half_deg, lat + half_deg],
        [lon - half_deg, lat - half_deg]]]}

TIME_RANGE = ("2025-01-01", "2025-12-31")  # one full year

print(f"{len(LAKES)} lakes, ~{len(LAKES) * 365 // 10} intervals")

In [ ]:
def lake_series(name, lat, lon):
    """(dates, mean_C, stdev_C) for one lake; empty lists if nothing usable."""
    req = SentinelHubStatistical(
        aggregation=SentinelHubStatistical.aggregation(
            evalscript=evalscript_lswt,
            time_interval=TIME_RANGE,
            aggregation_interval="P10D",  # the product's native dekad
            size=(8, 8),                  # ~6 km box, 1 km product
        ),
        input_data=[SentinelHubStatistical.input_data(lswt)],
        geometry=Geometry(box(lat, lon), crs=CRS.WGS84),
        config=config,
    )
    dates, temps, spread = [], [], []
    for item in req.get_data()[0]["data"]:
        if "outputs" not in item:
            continue
        band = item["outputs"]["celsius"]["bands"]["B0"]["stats"]
        if band["sampleCount"] <= band["noDataCount"]:
            continue
        dates.append(item["interval"]["from"][:10])
        temps.append(band["mean"])
        spread.append(band["stDev"])  # spatial spread across the lake, not error
    return dates, temps, spread


series = {}
for name, lat, lon in LAKES:
    try:
        d, t, sd = lake_series(name, lat, lon)
    except Exception as exc:  # one bad lake must not kill the run
        print(f"  {name:18s} failed: {type(exc).__name__}")
        continue
    if t:
        series[name] = (d, t, sd)
    print(f"  {name:18s} {len(t):3d} points{'' if t else '   <- check the box is on water'}")

print(f"\n{len(series)}/{len(LAKES)} lakes returned data")

In [ ]:
import matplotlib.dates as mdates
from datetime import date

lat_of = {name: lat for name, lat, _lon in LAKES}

fig, ax = plt.subplots(figsize=(12, 6))
for name in sorted(series, key=lambda n: -lat_of[n]):  # north to south
    d, t, sd = series[name]
    x = [date.fromisoformat(v) for v in d]
    line, = ax.plot(x, t, marker=".", ms=3, lw=1.2, label=name)
    ax.fill_between(x, [m - s for m, s in zip(t, sd)],
                       [m + s for m, s in zip(t, sd)],
                    color=line.get_color(), alpha=0.12, lw=0)

ax.set_title("Lake surface water temperature, 2025 — CLMS 1 km, 10-daily")
ax.set_ylabel("\u00b0C")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()

## §7 — Links

- CDSE custom scripts: https://github.com/eu-cdse/sentinel-hub-custom-scripts
- eo-learn / eo-grow: https://github.com/sentinel-hub/eo-learn · https://github.com/sentinel-hub/eo-grow
- Requests Builder (GUI to runnable Python): https://shapps.dataspace.copernicus.eu/requests-builder/